In [2]:


import numpy as np
import pandas as pd

SOFT_PATH = "dane/interim/fact_inka_soft_flagged_rotacja_wazone.parquet"
HARD_PATH = "dane/interim/fact_inka_hard_flagged_wazone.parquet"

SOFT_COLS = [
    "TowId", "Data", "DokId", "Kolejnosc", "NrDok", "TypDok", "TypRuchu",
    "IloscPlus", "IloscMinus", "CenaPoRab", "WplywNaStan", "MetodaLiczenia",
    "Mnoznik", "AktywnyDok", "NazwaTow", "NazwaAsort", "EAN", "AktywnyTow",
    "JestWazony", "CzyNiechciane",
]


def wczytaj_i_przygotuj_ruchy(path: str = SOFT_PATH) -> pd.DataFrame:
    """Wczytuje ruchy magazynowe i zostawia tylko te, które realnie wpływają
    na stan (WplywNaStan=True) i pochodzą z aktywnego dokumentu (AktywnyDok=1).
    Odrzuca też "puste" wiersze resetu (BO/remanent bez żadnej ilości)."""
    df = pd.read_parquet(path, columns=SOFT_COLS)

    df = df[(df["WplywNaStan"] == True) & (df["AktywnyDok"] == 1)].copy()

    is_reset_raw = df["MetodaLiczenia"] == "RESET_IP"
    pusty_reset = is_reset_raw & (df["IloscPlus"] == 0) & (df["IloscMinus"] == 0)
    df = df[~pusty_reset].copy()

    df = df.sort_values(["TowId", "Data", "DokId", "Kolejnosc"]).reset_index(drop=True)
    return df


def wylicz_stan_narastajaco(df: pd.DataFrame) -> pd.DataFrame:
    """Dla każdego towaru liczy stan narastająco w czasie, z resetami
    przy BO/remanencie. Dodaje kolumny: delta, is_reset, running_stock."""
    is_reset = df["MetodaLiczenia"] == "RESET_IP"

    delta = np.where(
        df["MetodaLiczenia"] == "IP",
        df["Mnoznik"] * (df["IloscPlus"] - df["IloscMinus"]),
        np.where(
            df["MetodaLiczenia"] == "IP_IM_DELTA",
            df["IloscPlus"] - df["IloscMinus"],
            0.0,
        ),
    )
    delta = np.where(is_reset, 0.0, delta)  # reset nie dodaje deltą, tylko nadpisuje

    df = df.copy()
    df["delta"] = delta
    df["is_reset"] = is_reset

    # numer "epoki" między kolejnymi resetami - w obrębie jednej epoki
    # sumujemy delty od wartości bazowej ustawionej resetem
    df["group"] = df.groupby("TowId")["is_reset"].cumsum()

    baseline = np.where(is_reset, df["IloscPlus"], np.nan)
    df["baseline"] = baseline
    df["baseline"] = df.groupby("TowId")["baseline"].ffill().fillna(0.0)

    df["cum_delta_in_group"] = df.groupby(["TowId", "group"])["delta"].cumsum()
    df["running_stock"] = df["baseline"] + df["cum_delta_in_group"]

    return df


def zbuduj_raport_stanow(df_z_running_stock: pd.DataFrame) -> pd.DataFrame:
    """Bierze ostatni wiersz każdego towaru = jego bieżący stan magazynowy."""
    ostatni = (
        df_z_running_stock
        .sort_values(["TowId", "Data", "DokId", "Kolejnosc"])
        .groupby("TowId")
        .tail(1)
    )
    raport = ostatni[[
        "TowId", "NazwaTow", "NazwaAsort", "EAN", "AktywnyTow",
        "JestWazony", "CzyNiechciane", "Data", "CenaPoRab", "running_stock",
    ]].copy()
    raport = raport.rename(columns={
        "Data": "DataOstatniegoRuchu",
        "running_stock": "StanMagazynowy",
        "CenaPoRab": "OstatniaCenaPoRab",
    })
    raport["WartoscMagazynowa"] = raport["StanMagazynowy"] * raport["OstatniaCenaPoRab"]
    return raport.sort_values("WartoscMagazynowa", ascending=False)


def dolacz_flage_martwy(raport: pd.DataFrame, hard_path: str = HARD_PATH) -> pd.DataFrame:
    """Dokleja do raportu flagę JestMartwy z pliku hard-flagged (stała per TowId)."""
    hard = pd.read_parquet(hard_path, columns=["TowId", "JestMartwy"])
    martwy = hard.groupby("TowId")["JestMartwy"].agg(lambda s: s.mode().iloc[0])
    return raport.merge(martwy.rename("Martwy"), on="TowId", how="left").fillna({"Martwy": 0})


def sprawdz_zgodnosc_z_bo_i_remanentem(df_z_running_stock: pd.DataFrame) -> pd.DataFrame:
    """Kontrola jakości: dla każdego zdarzenia RESET_IP (BO/remanent) porównuje
    stan WYLICZONY tuż przed nim (prev_stock) ze stanem SPISANYM w systemie
    (kolumna IloscMinus w wierszu resetu = stara ilość przed spisem).
    Zwraca tabelę zdarzeń z kolumną 'diff' (0 = zgodność)."""
    df = df_z_running_stock.copy()
    df["prev_stock"] = df.groupby("TowId")["running_stock"].shift(1).fillna(0.0)

    resety = df[df["is_reset"]].copy()
    resety["diff"] = resety["prev_stock"] - resety["IloscMinus"]
    return resety[[
        "TowId", "NazwaTow", "Data", "NrDok", "TypRuchu",
        "prev_stock", "IloscMinus", "diff", "IloscPlus",
    ]]


def main():
    ruchy = wczytaj_i_przygotuj_ruchy()
    ruchy = wylicz_stan_narastajaco(ruchy)

    raport = zbuduj_raport_stanow(ruchy)
    raport = dolacz_flage_martwy(raport)
    raport.to_parquet("stan_magazynowy.parquet", index=False)

    print(f"Produktów w raporcie: {len(raport)}")
    print(f"Łączna wartość magazynu: {raport['WartoscMagazynowa'].sum():,.2f} zł")

    kontrola = sprawdz_zgodnosc_z_bo_i_remanentem(ruchy)
    for typ in ("bo", "remanent"):
        cz = kontrola[kontrola["TypRuchu"] == typ]
        zgodne = (cz["diff"].abs() <= 0.001).sum()
        print(f"{typ}: {zgodne}/{len(cz)} zgodnych ({100*zgodne/len(cz):.1f}%)")

    kontrola.to_parquet("kontrola_bo_remanent.parquet", index=False)


if __name__ == "__main__":
    main()

Produktów w raporcie: 12481
Łączna wartość magazynu: 366,191.34 zł
bo: 5418/5418 zgodnych (100.0%)
remanent: 17828/18757 zgodnych (95.0%)


"""
Obliczanie stanu magazynowego na podstawie ruchów magazynowych ERP Inka.

Wejście:
    fact_inka_soft_flagged_rotacja_wazone.parquet
        - zawiera WSZYSTKIE typy ruchów (sprzedaż, przyjęcia, zwroty, remanenty,
          bilanse otwarcia BO, rozbieżności, przesunięcia, straty, przeceny, neutralne)
        - to jedyny plik, w którym są zdarzenia resetujące stan (BO, remanent),
          dlatego stan liczymy na jego podstawie

    fact_inka_hard_flagged_wazone.parquet
        - podzbiór samej sprzedaży (TypRuchu='sprzedaz'), wzbogacony o:
            JestMartwy          - flaga rotacji "martwy towar" (per TowId, stała w czasie)
            NazwaTowCleanName   - znormalizowana nazwa towaru
            NazwaAsortCleanName - znormalizowana nazwa kategorii
          używany tu tylko do domięszania tych kolumn do finalnego raportu

Logika liczenia stanu (kolumna po kolumnie z pliku źródłowego):
    WplywNaStan     - bool: czy wiersz w ogóle wpływa na stan magazynowy
                      (część wierszy, np. neutralny/przecena, tego nie robi)
    AktywnyDok      - 1/0: czy dokument jest aktywny. Wiersze AktywnyDok=0 to
                      zduplikowane/anulowane linie tego samego dokumentu
                      (np. część linii remanentów) - trzeba je odrzucić,
                      inaczej ten sam towar liczy się dwa razy
    MetodaLiczenia  - sposób naliczania zmiany stanu:
        'RESET_IP'      -> bilans otwarcia (BO) lub remanent: NIE dodaje się
                            do bieżącego stanu, tylko GO ZASTĘPUJE wartością
                            z kolumny IloscPlus (stan spisany na nowo)
        'IP'            -> zwykły ruch: delta = Mnoznik * (IloscPlus - IloscMinus)
                            (Mnoznik: +1 dla ruchów przychodowych, -1 dla rozchodowych)
        'IP_IM_DELTA'   -> ruch z korektą różnicy: delta = IloscPlus - IloscMinus
                            (rozbieżności inwentaryzacyjne, przesunięcia)
        'Brak'          -> wiersz nie wpływa na stan (pokrywa się z WplywNaStan=False)

    Osobny przypadek: "pusty" wiersz RESET_IP, w którym IloscPlus=0 ORAZ
    IloscMinus=0 (np. duplikat dokumentu BO/23/2 z 2023-01-01, powielający
    11 003 pozycje z BO/23/1, ale bez żadnych ilości) - taki wiersz nic nie
    wnosi i NIE powinien resetować dotychczas wyliczonego stanu do zera.
    Jest odrzucany przed liczeniem.

Wynik: jeden wiersz na TowId z bieżącym stanem, ostatnią ceną i wartością
magazynową, zapisany do stan_magazynowy.parquet.

Zawiera też funkcję kontrolną `sprawdz_zgodnosc_z_bo_i_remanentem`, która
weryfikuje, czy wyliczony stan TUŻ PRZED każdym zdarzeniem BO/remanentem
zgadza się z ilością spisaną w tym zdarzeniu (kolumna IloscMinus przy
MetodaLiczenia='RESET_IP' to "stara" ilość przed resetem).
"""